## 1. Загрузка и разделение на выборки

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/change/change_penguins.csv')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [2]:
df.isnull().sum()

species               0
island                0
bill_length_mm       36
bill_depth_mm         2
flipper_length_mm     2
body_mass_g          36
sex                  11
dtype: int64

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop('species', axis=1)
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Train: {len(X_train)} строк")
print(f"Test:  {len(X_test)} строк")
print(f"\nРаспределение классов в train:\n{y_train.value_counts()}")
print(f"\nРаспределение классов в test:\n{y_test.value_counts()}")

Train: 275 строк
Test:  69 строк

Распределение классов в train:
species
Adelie       122
Gentoo        99
Chinstrap     54
Name: count, dtype: int64

Распределение классов в test:
species
Adelie       30
Gentoo       25
Chinstrap    14
Name: count, dtype: int64


## 2. ColumnTransformer, CustomTransformer

In [4]:
def replace_rare_categories(df, threshold=0.05):
    df_replace = df.copy()
    for col in df_replace.columns:
        freq = df_replace[col].value_counts(normalize=True)
        rare_cats = freq[freq < threshold].index.tolist()
        df_replace[col] = df_replace[col].apply(lambda x: 'Other' if x in rare_cats else x)
    return df_replace

In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer


CustomTransformer = FunctionTransformer(replace_rare_categories)

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn import set_config


set_config(transform_output="pandas")

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('rare', CustomTransformer),
    ('onehot', OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']),
    ('cat', cat_pipe, ['island', 'sex'])
])

In [7]:
from sklearn.ensemble import RandomForestClassifier


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 7, None]
}


grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Лучшие параметры: {grid.best_params_}")
print(f"Лучший CV score:  {grid.best_score_:.3f}")

Лучшие параметры: {'model__max_depth': None, 'model__n_estimators': 200}
Лучший CV score:  0.982


In [9]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

y_pred_grid = grid.predict(X_test)
print(f"Test accuracy:    {accuracy_score(y_test, y_pred_grid)}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9420289855072463
Test accuracy:    0.9420289855072463

Classification Report:
              precision    recall  f1-score   support

      Adelie       0.91      0.97      0.94        30
   Chinstrap       0.92      0.79      0.85        14
      Gentoo       1.00      1.00      1.00        25

    accuracy                           0.94        69
   macro avg       0.94      0.92      0.93        69
weighted avg       0.94      0.94      0.94        69



In [11]:
import joblib


joblib.dump(grid, '../models/grid_search_model.pkl')

['../models/grid_search_model.pkl']